# PoRep Market — State Inspector  _(read-only)_

Query the deployed contracts without sending any transactions.
Useful for debugging a live deployment or verifying happy-path postconditions.

```
pip install web3 python-dotenv
```
Set `RPC_URL` env var or edit `CONFIG` below.

In [ ]:
import json, os
from pathlib import Path
from dotenv import load_dotenv
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

load_dotenv()

CONFIG = {
    "rpc_url":           os.getenv("RPC_URL", "http://127.0.0.1:1234/rpc/v1"),
    "porep_market":      "0xD20Fc289be410d7CBF3fFF2e3d83778ea21c3b7B",
    "sp_registry":       "0xc66fc2b7FD6aa03047e43a6CA2E23C2296CCddAA",
    "validator_factory": "0x6CCCE1D85e75DC7B402e70f5D2aB6544D792dCb5",
    "sli_oracle":        "0x6948607aDbB10AA51D298E7759Df35564145da32",
    "sli_scorer":        "0xb5A34880C80A7bFb4159ab0E9Ce0dbEbbB2C3807",
}

ABI_DIR = Path("../abis")
def load_abi(name): return json.loads((ABI_DIR / f"{name}.json").read_text())

w3 = Web3(Web3.HTTPProvider(CONFIG["rpc_url"]))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
assert w3.is_connected(), f"Cannot reach {CONFIG['rpc_url']}"

porep_market     = w3.eth.contract(address=CONFIG["porep_market"],      abi=load_abi("PoRepMarket"))
sp_registry      = w3.eth.contract(address=CONFIG["sp_registry"],       abi=load_abi("SPRegistry"))
validator_factory = w3.eth.contract(address=CONFIG["validator_factory"], abi=load_abi("ValidatorFactory"))
sli_oracle       = w3.eth.contract(address=CONFIG["sli_oracle"],         abi=load_abi("SLIOracle"))

DEAL_STATES = {0: "Proposed", 1: "Accepted", 2: "Completed", 3: "Rejected", 4: "Terminated"}
print(f"Connected  chain={w3.eth.chain_id}  block={w3.eth.block_number}")

### SP Registry — all providers

In [ ]:
providers = sp_registry.functions.getProviders().call()
committed_providers = sp_registry.functions.getCommittedProviders().call()
print(f"Total registered : {len(providers)}")
print(f"With committed capacity : {len(committed_providers)}")

for actor_id in providers:
    info = sp_registry.functions.getProviderInfo(actor_id).call()
    org, payee, paused, blocked, caps, avail, committed, pending, price = info
    flags = " ".join(f for f, v in [("PAUSED", paused), ("BLOCKED", blocked)] if v) or "active"
    print(f"\n  [{flags}] Actor {actor_id}")
    print(f"    org={org}  payee={payee}")
    print(f"    avail={avail:>20,}  committed={committed:>20,}  pending={pending:>20,}")
    print(f"    price/sector/month={price}")
    print(f"    SLI caps: ret={caps[0]}bps  bw={caps[1]}Mbps  lat={caps[2]}ms  idx={caps[3]}%")

### SLI Oracle — latest attestations

In [ ]:
providers = sp_registry.functions.getProviders().call()
for actor_id in providers:
    att = sli_oracle.functions.getAttestation(actor_id).call()
    last_update, slis = att
    blocks_ago = w3.eth.block_number - last_update if last_update else "never"
    print(f"Actor {actor_id}  lastUpdate=block {last_update} ({blocks_ago} blocks ago)")
    print(f"  ret={slis[0]}bps  bw={slis[1]}Mbps  lat={slis[2]}ms  idx={slis[3]}%")

### PoRepMarket — completed deals (settlement queue)

In [ ]:
completed = porep_market.functions.getCompletedDeals().call()
print(f"Deals in settlement queue: {len(completed)}")
for d in completed:
    print(f"  dealId={d['dealId']}  provider={d['provider']}  railId={d['railId']}  "
          f"validator={d['validator']}")

### PoRepMarket — inspect specific deal

In [ ]:
# Set the deal ID you want to inspect
DEAL_ID = 1

try:
    proposal = porep_market.functions.getDealProposal(DEAL_ID).call()
except Exception as e:
    print(f"Deal {DEAL_ID} not found: {e}")
    proposal = None

if proposal:
    state_str = DEAL_STATES.get(proposal["state"], str(proposal["state"]))
    print(f"dealId           : {proposal['dealId']}")
    print(f"state            : {state_str}")
    print(f"client           : {proposal['client']}")
    print(f"provider (actor) : {proposal['provider']}")
    print(f"validator        : {proposal['validator']}")
    print(f"railId           : {proposal['railId']}")
    print(f"manifestLocation : {proposal['manifestLocation']}")
    t = proposal["terms"]
    print(f"terms            : size={t['dealSizeBytes']:,} bytes  "
          f"price={t['pricePerSectorPerMonth']}  duration={t['durationDays']}d")
    r = proposal["requirements"]
    print(f"requirements     : ret={r['retrievabilityBps']}bps  "
          f"bw={r['bandwidthMbps']}Mbps  lat={r['latencyMs']}ms  idx={r['indexingPct']}%")

    # If a validator is deployed, inspect it
    if proposal["validator"] != "0x" + "0"*40:
        validator = w3.eth.contract(address=proposal["validator"], abi=load_abi("Validator"))
        min_epochs = validator.functions.getMinEpochsBetweenSettlements().call()
        print(f"\nValidator minEpochsBetweenSettlements : {min_epochs}")

### ValidatorFactory — check if validator exists for a deal

In [ ]:
# Query multiple deal IDs at once
deal_ids_to_check = list(range(1, 6))  # check deals 1-5

for did in deal_ids_to_check:
    try:
        addr = validator_factory.functions.getInstance(did).call()
        exists = validator_factory.functions.isValidatorContract(addr).call()
        code = w3.eth.get_code(addr)
        deployed = len(code) > 2
        print(f"Deal {did:>3} : validator={addr}  isValidator={exists}  deployed={deployed}")
    except Exception:
        print(f"Deal {did:>3} : no validator")